# Analisi delle Note Spese

Questo notebook dimostra come creare agenti che utilizzano plugin per elaborare le spese di viaggio a partire da immagini locali di ricevute, generare un'email per il rimborso spese e visualizzare i dati delle spese con un grafico a torta. Gli agenti scelgono dinamicamente le funzioni in base al contesto del compito.

Passaggi:
1. L'agente OCR elabora l'immagine locale della ricevuta ed estrae i dati delle spese di viaggio.
2. L'agente Email genera un'email per la richiesta di rimborso spese.

### Esempio di scenario di spese di viaggio:
Immagina di essere un dipendente in viaggio per una riunione di lavoro in un'altra città. La tua azienda ha una politica per rimborsare tutte le spese ragionevoli relative al viaggio. Ecco una suddivisione delle potenziali spese di viaggio:
- Trasporto:
Biglietto aereo per un viaggio di andata e ritorno dalla tua città di origine alla città di destinazione.
Taxi o servizi di ride-hailing per e dall'aeroporto.
Trasporti locali nella città di destinazione (come trasporto pubblico, auto a noleggio o taxi).

- Alloggio:
Soggiorno in hotel per tre notti in un hotel business di fascia media vicino al luogo della riunione.

- Pasti:
Indennità giornaliera per colazione, pranzo e cena, basata sulla politica di diaria dell'azienda.

- Spese Varie:
Tariffe di parcheggio all'aeroporto.
Costi di accesso a Internet in hotel.
Mance o piccoli costi per il servizio.

- Documentazione:
Invi di tutte le ricevute (voli, taxi, hotel, pasti, ecc.) e un modulo di rimborso spese compilato per il rimborso.


## Importa le librerie necessarie

Importa le librerie e i moduli necessari per il notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import Content, Message, tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Definire i Modelli di Spesa

 Crea un modello Pydantic per le spese individuali e una classe ExpenseFormatter per convertire una query utente in dati di spesa strutturati.

 Ogni spesa sarà rappresentata nel formato:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Definizione degli Strumenti - Generazione dell'Email

Crea una funzione strumento per generare un'email per l'invio di una richiesta di rimborso spese.
- Questo strumento utilizza il decoratore `@tool` del Microsoft Agent Framework.
- Calcola l'importo totale delle spese e formatta i dettagli nel corpo dell'email.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Caricamento delle immagini delle ricevute di spese di viaggio

Crea una funzione di supporto per caricare le immagini delle ricevute come contenuto multimodale nativo.
- Legge i byte dell'immagine della ricevuta e li racchiude nel `Content` di Agent Framework affinché l'agente li analizzi.


In [ ]:
def load_receipt_image(image_path: str = "receipt.jpg") -> Content:
    """Load a receipt image as native multimodal content."""
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    return Content.from_data(image_bytes, "image/jpeg")

## Spese di elaborazione

Definisci gli agenti e collegali in un flusso di lavoro sequenziale usando `WorkflowBuilder`.
- L'agente OCR estrae dati di spesa strutturati dall'immagine della ricevuta fornita nel messaggio dell'utente.
- L'agente Email prende i dati estratti e genera un'email professionale per la richiesta di rimborso spese usando lo strumento `generate_expense_email`.
- `WorkflowBuilder` con `add_edge` crea una pipeline sequenziale: Agente OCR → Agente Email.


In [ ]:
ocr_agent = client.as_agent(
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Analyze the receipt image supplied in the user message and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Funzione principale

Costruisci il flusso di lavoro sequenziale ed eseguilo per elaborare l'immagine della ricevuta e generare l'email di richiesta spese.


> **Nota:** I byte della ricevuta sono incapsulati nel multimodale `Content` nativo di Agent Framework, quindi Foundry li riceve come input immagine anziché come testo ordinario.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
receipt_message = Message(
    role="user",
    contents=[prompt, load_receipt_image("receipt.jpg")],
)

events = workflow.run(
    receipt_message,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
